# Векторизация

In [2]:
import pandas as pd
import numpy as np
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer
import nltk

from sentence_transformers import SentenceTransformer

df = pd.read_csv("../data/split/02_text_with_metrics.csv")

print(len(df))
df.head(2)

c:\Users\enoni\PycharmProjects\ai_channels_feature_engineering\emb_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


10479


,channel_username,post_id,text,text_clean,text_ruts,text_length,text_length_clean,links_count_entities,has_links_entities,links_from_entities,n_words,n_sents,n_chars,flesch_reading_ease,flesch_kincaid_grade,ttr,words_per_sent,flesch_reading_ease_capped,flesch_kincaid_grade_capped
0,AI_point_of_view,1813,🤔 Искусственный интеллект: угроза или новые во...,искусственный интеллект: угроза или новые возм...,Искусственный интеллект: угроза или новые возм...,1173,148,0,False,[],146.0,8.0,1163.0,8.573014,13.552500,0.821918,18.250000,8.573014,13.5525
1,AI_point_of_view,1809,👩‍🎓 **Самообучение без разметки: лекция Алексе...,самообучение без разметки: лекция алексея зайц...,Самообучение без разметки: лекция Алексея Зайц...,1661,213,0,False,[],215.0,6.0,1597.0,9.582364,19.269264,0.772093,35.833333,9.582364,18.1000


Проверим датасет на наличие пустых текстов

In [30]:
print("NaN text_cleaned:", df["text_clean"].isna().sum())

NaN text_cleaned: 12


In [31]:
empty_mask = df["text_clean"].isna() | (df["text_clean"].str.strip() == "")

empty_texts = df[empty_mask]

print("Количество пустых текстов:", len(empty_texts))
empty_texts

Количество пустых текстов: 12


,channel_username,post_id,text,text_clean,text_ruts,text_length,text_length_clean,links_count_entities,has_links_entities,links_from_entities,n_words,n_sents,n_chars,flesch_reading_ease,flesch_kincaid_grade,ttr,words_per_sent,flesch_reading_ease_capped,flesch_kincaid_grade_capped
7536,seeallochnaya,3151,😕🔪,NaN,NaN,2,0,0,False,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7689,seeallochnaya,2944,NaN,NaN,NaN,48,0,1,False,['https://www.anthropic.com/news/claude-sonnet...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7738,seeallochnaya,2868,🤯,NaN,NaN,1,0,0,False,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7847,seeallochnaya,2724,👀 👀 👀,NaN,NaN,77,0,1,False,['https://cdn.openai.com/API/docs/images/model...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7939,seeallochnaya,2600,👆,NaN,NaN,1,0,0,False,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7959,seeallochnaya,2578,👆,NaN,NaN,1,0,0,False,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8086,seeallochnaya,2384,👆,NaN,NaN,1,0,0,False,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8094,seeallochnaya,2372,NaN,NaN,NaN,46,0,1,False,['https://cdn.openai.com/gpt-4-5-system-card.p...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8098,seeallochnaya,2367,🥰,NaN,NaN,1,0,0,False,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8202,seeallochnaya,2246,😦,NaN,NaN,1,0,0,False,[],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Видим, что это просто эмодзи (скорее всего подписи к фото), поэтому на данном этапе удалим их

In [33]:
df = df.copy()

df = df[df["text_clean"].notna()]
df = df[df["text_clean"].str.strip() != ""]
df.reset_index(drop=True, inplace=True)

Подготовка текстов 

для TF-IDF

In [38]:
russian_stopwords = set(stopwords.words('russian'))
english_stopwords = set(stopwords.words('english'))
stemmer = SnowballStemmer("russian")

english_service = {'and', 'are', 'for', 'that', 'the', 'this', 'not', 'just',
                   'one', 'new', 'you', 'what', 'with', 'to', 'it', 'be', 'go',
                   'from', 'your', 'has', 'had', 'was', 'were', 'been', 'being'}
                   
def prepare_for_tfidf(text):
    if not isinstance(text, str):
        return ""
    
    text = re.sub(r'\d+', ' ', text)
    
    words = text.split()
    # убираем стоп-слова
    words = [w for w in words if w not in russian_stopwords.union(english_stopwords) and len(w) > 2]
    
    # стемминг
    words = [stemmer.stem(w) for w in words]
    
    return " ".join(words)

df["text_tfidf"] = df["text_clean"].apply(prepare_for_tfidf)

In [132]:
df[["text_clean", "text_tfidf"]].head(2)

,text_clean,text_tfidf
0,искусственный интеллект: угроза или новые возм...,искусственный интеллект: угроза новые возможно...
1,самообучение без разметки: лекция алексея зайц...,самообучение разметки: лекция алексея зайцева ...


Для моделей

In [39]:
def prepare_for_transformer(text):
    """Минимум: lowercase, убрать лишние пробелы."""
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df["text_transformer"] = df["text_clean"].apply(prepare_for_transformer)

In [7]:
df[["text_clean", "text_transformer"]].head()

,text_clean,text_transformer
0,искусственный интеллект: угроза или новые возм...,искусственный интеллект: угроза или новые возм...
1,самообучение без разметки: лекция алексея зайц...,самообучение без разметки: лекция алексея зайц...
2,искусственный интеллект: как научить машину чу...,искусственный интеллект: как научить машину чу...
3,"физика ии от сколтеха владимир вановский, руко...","физика ии от сколтеха владимир вановский, руко..."
4,завтра: sber conf: open source ai agents! сбер...,завтра: sber conf: open source ai agents! сбер...


In [47]:
columns_to_keep = [
    'channel_username', 'post_id',
    'text', 'text_clean', 'text_tfidf', 'text_transformer'
]

existing_cols = [col for col in columns_to_keep if col in df.columns]
df_final = df[existing_cols].copy()

output_path = "../data/split/02_text_for_vectorization.csv"
df_final.to_csv(output_path, index=False)

## Реализация TF-IDF

In [48]:
tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.6,
    sublinear_tf=True
)

X_tfidf = tfidf.fit_transform(df["text_tfidf"])

print("TF-IDF shape:", X_tfidf.shape)

TF-IDF shape: (10467, 8000)


In [ ]:
np.save("../data/embeddings/tfidf.npy", X_tfidf)

# all-MiniLM

In [ ]:
model_sbert = SentenceTransformer("all-MiniLM-L6-v2")

X_sbert = model_sbert.encode(
    df["text_transformer"].tolist(),
    show_progress_bar=True,
    batch_size=32
)

print("SBERT shape:", X_sbert.shape)

Batches:   0%|          | 0/328 [00:00<?, ?it/s]

SBERT shape: (10467, 384)


In [ ]:
np.save("../data/embeddings/sbert.npy", X_sbert)

# bge-m3

In [ ]:
texts_bge = df["text_transformer"].tolist()

model_bge = SentenceTransformer("BAAI/bge-m3")

X_bge = model_bge.encode(
    texts_bge,
    show_progress_bar=True,
    batch_size=64,
    normalize_embeddings=True
)

print("BGE-M3 shape:", X_bge.shape)

Batches:   0%|          | 0/164 [00:00<?, ?it/s]

BGE-M3 shape: (10467, 1024)


In [ ]:
np.save("../data/embeddings/bge.npy", X_bge)

# E5

In [ ]:
texts_e5 = [
    "Instruct: Represent the post for semantic clustering. Passage: " + t
    for t in df["text_transformer"]
]

model_e5 = SentenceTransformer("intfloat/multilingual-e5-large-instruct")

X_e5 = model_e5.encode(
    texts_e5,
    show_progress_bar=True,
    batch_size=64,
    normalize_embeddings=True
)

print("E5 shape:", X_e5.shape)

c:\Users\enoni\PycharmProjects\ai_channels_feature_engineering\emb_env\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Batches: 100%|██████████| 164/164 [3:27:58<00:00, 76.09s/it]  

E5 shape: (10467, 1024)


In [52]:
len(X_e5)

10467

In [53]:
np.save("../data/embeddings/e5.npy", X_e5)

Проверка эмбеддингов

In [3]:
import os
from scipy.sparse import issparse

def load_embedding(filepath, name):
    """Загружает эмбеддинги из .npy файла и выводит статистику."""
    if not os.path.exists(filepath):
        print(f"[ПРОПУСК] {name}: файл не найден ({filepath})")
        return None
    
    # Загружаем с allow_pickle для совместимости
    loaded = np.load(filepath, allow_pickle=True)
    
    # Если загруженный объект - numpy массив нулевой размерности, 
    # внутри которого лежит другой объект, извлекаем его
    if isinstance(loaded, np.ndarray) and loaded.ndim == 0:
        X = loaded.item()
    else:
        X = loaded
    
    # Приводим к dense для проверки статистик
    if issparse(X):
        X_dense = X.toarray()
    elif isinstance(X, np.ndarray):
        X_dense = X
    else:
        print(f"[ОШИБКА] {name}: неизвестный тип данных: {type(X)}")
        return None
    
    # Приводим к float для безопасного вычисления норм
    X_dense = X_dense.astype(np.float64)
    
    norms = np.linalg.norm(X_dense, axis=1)
    
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(f"  Shape:         {X_dense.shape}")
    print(f"  NaN count:     {np.isnan(X_dense).sum()}")
    print(f"  Inf count:     {np.isinf(X_dense).sum()}")
    print(f"  Mean norm:     {np.mean(norms):.4f}")
    print(f"  Std norm:      {np.std(norms):.4f}")
    print(f"  Min norm:      {np.min(norms):.6f}")
    print(f"  Max norm:      {np.max(norms):.4f}")
    
    return X

embeddings_dir = "../data/embeddings"

embeddings = {}

# TF-IDF + SVD
embeddings["TF-IDF"] = load_embedding(
    os.path.join(embeddings_dir, "tfidf.npy"),
    "TF-IDF"
)

# SBERT
embeddings["SBERT"] = load_embedding(
    os.path.join(embeddings_dir, "sbert.npy"),
    "SBERT (all-MiniLM-L6-v2)"
)

# BGE
embeddings["BGE"] = load_embedding(
    os.path.join(embeddings_dir, "bge.npy"),
    "BGE-M3"
)

# E5
embeddings["E5"] = load_embedding(
    os.path.join(embeddings_dir, "e5.npy"),
    "E5 (multilingual-e5-large-instruct)"
)


TF-IDF
  Shape:         (10467, 8000)
  NaN count:     0
  Inf count:     0
  Mean norm:     0.9997
  Std norm:      0.0169
  Min norm:      0.000000
  Max norm:      1.0000

SBERT (all-MiniLM-L6-v2)
  Shape:         (10467, 384)
  NaN count:     0
  Inf count:     0
  Mean norm:     1.0000
  Std norm:      0.0000
  Min norm:      1.000000
  Max norm:      1.0000

BGE-M3
  Shape:         (10467, 1024)
  NaN count:     0
  Inf count:     0
  Mean norm:     1.0000
  Std norm:      0.0000
  Min norm:      1.000000
  Max norm:      1.0000

E5 (multilingual-e5-large-instruct)
  Shape:         (10467, 1024)
  NaN count:     0
  Inf count:     0
  Mean norm:     1.0000
  Std norm:      0.0000
  Min norm:      1.000000
  Max norm:      1.0000
